# **Dataset Maker**

Follow this guide to make your own custom dataset using mostly defaultand previously tested parameters for all steps. 

**Discalimer:** It is recomended to have a copy of your data to start over since some of these steps are not revertible :O (even though no destructive editing to your data is applied).

## **1. Dataset Structure and Configuration**

### **Structure**

Before running any code you must make sure your data is organized as below. 

```
main/
├── raw/
│   ├── train/
│   │   ├── audio1.wav
│   │   └── ...
│   └── validation/
│       ├── audio2.wav
│       └── ...
└── config/
    └── dataprep.json
```

**Note:** Observe that you can completely disregard train/validation split by simply placing all your audio inside of the `raw/` directory.

### **Configuration**

You can create a the configuration file `config/dataprep.json` by simple creating a `.txt` file,  filling it with the text below, and then change its extension to `.json`.

```json
{
  "atoms_frames": 21,
  "atoms_overlap_frames": 3,
  "train_split": "train",
  "val_split": "validation"
}
```

**Note:** You can eliminate train_split and val_split entries if you don't have premade splits.

## **2. Data Preparation**

From here, the data will be prepared in many steps. Make sure you write the path to your dataset in the following cell and then run the whole thing one by one. Some cells may take some time :)

### **2.1. Fix your dataset path**

In [ ]:
# Make repo root importable for this session (zero packaging)
import sys, pathlib
repo_root = pathlib.Path.cwd().parent
sys.path.insert(0, str(repo_root))
%load_ext autoreload
%autoreload 2

# Fill your dataset path here
path_to_dataset = "/path/to/dataset"

### **2.2. Precompute audio representation**

In [ ]:
from SCAPES.data.dataprep import atoms_maker

atoms_maker(path_to_dataset)

### **2.3. Initialize the dataset**

In [ ]:
from SCAPES.data.dataset import AtomSequenceDataset
from SCAPES.auxiliar.encodec_wrapper import EncodecProcessor
import torch

# Pick your device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Initialize 48kHz processor
processor_48k_streamable = EncodecProcessor(sr=48000, streamable=True, device=device)

# 1. Default configuration
segment_length, context_length, hop_size = 5, 5, 1

# 2. Setup the main dataset with the new opt-in requested_keys
dataset = AtomSequenceDataset(
    dataset_path=path_to_dataset, 
    segment_length=segment_length,
    context_length=context_length,
    hop_size=hop_size,
    requested_keys=["latent_past", "latent_present", "latent_context_win", 
                    "scale_past", "scale_present", "scale_context_win", "index"],
    device=device)

### **2.4. Split your data into training and validation**

**Note:** If you didn't have a custom split in the first step, use option 2 in the following cell.

In [ ]:
# # ------------------------------------------------------------
# # Option 1: your data is splitted and the splits are annotated in your dataprep.json
# # ------------------------------------------------------------
dataset.make_split()

# # ------------------------------------------------------------
# # Option 2: your data is not splitted so you want to create a random split with a certain percentage of validation data
# # ------------------------------------------------------------
# dataset.make_split(val_split=0.1)


### **2.5. Precompute semantic annotations**

In [ ]:
from SCAPES.data.dataprep import precompute_annotations
from SCAPES.models.factorization import load_global_encoder
from SCAPES.auxiliar.clap_wrapper import CLAPWrapper

method = "clap"
model = CLAPWrapper(version="2023", use_cuda=True)
precompute_annotations(
    dataset=dataset, 
    annotation_type=method, # "clap" or "custom"
    time_part="context_win",  # "past" or "context_win"
    model=model, 
    batch_size=128,
    device="cuda"
)

**Note:** If you want to explore your dataset before going forward. You can use the notebook in slowstart instead :)